In [0]:
df = spark.read.csv("/Volumes/databricks/home/analyticsgcp1/shriomgulia5/my_files/raw_prices.csv", header=True, inferSchema=True)
df.show()

+---+--------+---------+-----------+
| id|    coin|price_usd|  timestamp|
+---+--------+---------+-----------+
|  1| bitcoin|  77288.0|1.7892785E9|
|  2|ethereum|  2522.55|1.7892785E9|
|  3| bitcoin|  77288.0|1.7892785E9|
|  4|ethereum|  2522.55|1.7892785E9|
|  5| bitcoin|  77288.0|1.7892785E9|
|  6|ethereum|  2522.55|1.7892785E9|
|  7| bitcoin|  77288.0|1.7892785E9|
|  8|ethereum|  2522.55|1.7892785E9|
|  9| bitcoin|  77288.0|1.7892785E9|
| 10|ethereum|  2522.55|1.7892785E9|
| 11| bitcoin|  77288.0|1.7892785E9|
| 12|ethereum|  2522.72|1.7892785E9|
| 13| bitcoin|  77288.0|1.7892785E9|
| 14|ethereum|  2522.72|1.7892785E9|
| 15|  status|     NULL|1.7892785E9|
| 16|  status|     NULL|1.7892785E9|
| 17|  status|     NULL|1.7892785E9|
| 18|  status|     NULL|1.7892785E9|
| 19|  status|     NULL|1.7892785E9|
| 20|  status|     NULL|1.7892786E9|
+---+--------+---------+-----------+
only showing top 20 rows


In [0]:
df.count()

4139

In [0]:
df_clean = df.filter(df.price_usd.isNotNull())
df_clean.show()
print(f"Original rows: {df.count()}, Clean rows: {df_clean.count()}")

+---+--------+---------+-----------+
| id|    coin|price_usd|  timestamp|
+---+--------+---------+-----------+
|  1| bitcoin|  77288.0|1.7892785E9|
|  2|ethereum|  2522.55|1.7892785E9|
|  3| bitcoin|  77288.0|1.7892785E9|
|  4|ethereum|  2522.55|1.7892785E9|
|  5| bitcoin|  77288.0|1.7892785E9|
|  6|ethereum|  2522.55|1.7892785E9|
|  7| bitcoin|  77288.0|1.7892785E9|
|  8|ethereum|  2522.55|1.7892785E9|
|  9| bitcoin|  77288.0|1.7892785E9|
| 10|ethereum|  2522.55|1.7892785E9|
| 11| bitcoin|  77288.0|1.7892785E9|
| 12|ethereum|  2522.72|1.7892785E9|
| 13| bitcoin|  77288.0|1.7892785E9|
| 14|ethereum|  2522.72|1.7892785E9|
| 21| bitcoin|  77294.0|1.7892786E9|
| 22|ethereum|  2522.65|1.7892786E9|
| 23| bitcoin|  77294.0|1.7892786E9|
| 24|ethereum|  2522.65|1.7892786E9|
| 25| bitcoin|  77294.0|1.7892786E9|
| 26|ethereum|  2522.65|1.7892786E9|
+---+--------+---------+-----------+
only showing top 20 rows
Original rows: 4139, Clean rows: 2748


In [0]:
from pyspark.sql import functions as F

summary = df_clean.groupBy("coin").agg(
    F.avg("price_usd").alias("avg_price"),
    F.max("price_usd").alias("max_price"),
    F.min("price_usd").alias("min_price"),
    F.count("price_usd").alias("count")
)
summary.show()

+--------+------------------+---------+---------+-----+
|    coin|         avg_price|max_price|min_price|count|
+--------+------------------+---------+---------+-----+
| bitcoin| 77234.04148471616|  78256.0|  76334.0| 1374|
|ethereum|2501.2368995633124|  2530.38|  2436.59| 1374|
+--------+------------------+---------+---------+-----+



In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze.raw_prices")
df_clean.write.format("delta").mode("overwrite").saveAsTable("silver.raw_prices")
summary.write.format("delta").mode("overwrite").saveAsTable("gold.raw_prices")